[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/84_multimodal_rope_solution.ipynb)

# 🔴 Solution: Multimodal RoPE (M-RoPE)

Reference solution for `multimodal_rope`. The three coordinate planes are temporal, height, and width. Text tokens use the same value on all three planes, so this implementation becomes standard 1D RoPE for text.


In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def apply_multimodal_rope(
    q: torch.Tensor,
    k: torch.Tensor,
    position_ids: torch.Tensor,
    sections: tuple[int, int, int],
    base: float = 10000.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    if q.ndim != 4 or k.ndim != 4:
        raise ValueError('q and k must have shape (B, H, N, D)')

    batch, _, seq_len, head_dim = q.shape
    if k.shape[0] != batch or k.shape[2:] != (seq_len, head_dim):
        raise ValueError('q and k must share batch, sequence, and head dimensions')
    if q.device != k.device or q.dtype != k.dtype:
        raise ValueError('q and k must share device and dtype')
    if not q.is_floating_point() or head_dim % 2 != 0:
        raise ValueError('q and k must be floating point with an even head_dim')
    if position_ids.shape != (3, batch, seq_len):
        raise ValueError('position_ids must have shape (3, B, N)')
    if len(sections) != 3 or any(not isinstance(size, int) or size < 0 for size in sections):
        raise ValueError('sections must contain three non-negative integers')

    num_pairs = head_dim // 2
    if sum(sections) != num_pairs:
        raise ValueError('sections must sum to head_dim // 2')
    if base <= 0:
        raise ValueError('base must be positive')

    pair_axes = torch.cat([
        torch.full((size,), axis, device=q.device, dtype=torch.long)
        for axis, size in enumerate(sections)
    ])
    freq_dtype = torch.float64 if q.dtype == torch.float64 else torch.float32
    pair_index = torch.arange(num_pairs, device=q.device, dtype=freq_dtype)
    inv_freq = torch.pow(
        torch.tensor(base, device=q.device, dtype=freq_dtype),
        -2.0 * pair_index / head_dim,
    )

    positions = position_ids.to(device=q.device, dtype=freq_dtype)
    positions_by_pair = positions.index_select(0, pair_axes).permute(1, 2, 0)
    angles = positions_by_pair * inv_freq
    cos = angles.cos().to(q.dtype).unsqueeze(1)
    sin = angles.sin().to(q.dtype).unsqueeze(1)

    def rotate(x: torch.Tensor) -> torch.Tensor:
        pairs = x.reshape(*x.shape[:-1], num_pairs, 2)
        even, odd = pairs[..., 0], pairs[..., 1]
        rotated = torch.stack(
            (even * cos - odd * sin, even * sin + odd * cos),
            dim=-1,
        )
        return rotated.flatten(-2)

    return rotate(q), rotate(k)


In [ ]:
# Verify text fallback and a small video grid
q = torch.randn(1, 4, 8, 12)
k = torch.randn(1, 2, 8, 12)
t = torch.arange(2).repeat_interleave(4)
y = torch.arange(2).repeat_interleave(2).repeat(2)
x = torch.arange(2).repeat(2).repeat(2)
position_ids = torch.stack((t, y, x)).unsqueeze(1)
q_rot, k_rot = apply_multimodal_rope(q, k, position_ids, (2, 2, 2))
print(q_rot.shape, k_rot.shape)
print('norm preserved:', torch.allclose(q_rot.norm(dim=-1), q.norm(dim=-1), atol=1e-5))


In [ ]:
# Run judge
from torch_judge import check
check('multimodal_rope')
